In [3]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

1. Load the dataset

In [4]:
from ucimlrepo import fetch_ucirepo

adult = fetch_ucirepo(id=2)
data = pd.concat([adult.data.features, adult.data.targets], axis=1)

2. Clean the data - remove rows with missing values (marked as '?')

In [5]:
data = data.replace('?', np.nan)
data = data.dropna()

print("Dataset Info:")
print(f"Total samples: {len(data)}")
print(f"Features: {data.shape[1] - 1}")
print(f"Target variable: income")

Dataset Info:
Total samples: 45222
Features: 14
Target variable: income


3. Separate features and target, numeric and categorical columns

In [6]:
X = data.drop('income', axis=1)
y = data['income']

numeric_columns = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_columns = ['workclass', 'education', 'marital-status', 'occupation', 
                      'relationship', 'race', 'sex', 'native-country']

4. Print "Before Transformation Stats"

In [7]:
print("\n" + "="*60)
print("BEFORE TRANSFORMATION")
print("="*60)
print(f"Dataset shape: {X.shape}")
print(f"Numeric columns ({len(numeric_columns)}): {numeric_columns}")
print(f"Categorical columns ({len(categorical_columns)}): {categorical_columns}")

print("\nNumeric columns statistics (before scaling):")
print(X[numeric_columns].describe().round(2))

print("\nCategorical columns unique values:")
for col in categorical_columns:
    print(f"{col}: {X[col].nunique()} unique values")


BEFORE TRANSFORMATION
Dataset shape: (45222, 14)
Numeric columns (6): ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
Categorical columns (8): ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']

Numeric columns statistics (before scaling):
            age      fnlwgt  education-num  capital-gain  capital-loss  \
count  45222.00    45222.00       45222.00      45222.00      45222.00   
mean      38.55   189734.73          10.12       1101.43         88.60   
std       13.22   105639.20           2.55       7506.43        404.96   
min       17.00    13492.00           1.00          0.00          0.00   
25%       28.00   117388.25           9.00          0.00          0.00   
50%       37.00   178316.00          10.00          0.00          0.00   
75%       47.00   237926.00          13.00          0.00          0.00   
max       90.00  1490400.00          16.00      99999.00       4356.00   

5. Create preprocessing pipeline

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_columns)
    ],
    remainder='drop'  # Drop any columns not specified
)

6. Fit the preprocessor on training data and transform both train and test

In [9]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

7. Print "After Transformation" stats

In [10]:
print("\n" + "="*60)
print("AFTER TRANSFORMATION")
print("="*60)
print(f"Training set shape: {X_train_transformed.shape}")
print(f"Test set shape: {X_test_transformed.shape}")
print(f"Features increased from {X.shape[1]} to {X_train_transformed.shape[1]}")

numeric_feature_names = numeric_columns
categorical_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_columns)
all_feature_names = list(numeric_feature_names) + list(categorical_feature_names)

print(f"\nFeature breakdown:")
print(f"- Numeric features (scaled): {len(numeric_feature_names)}")
print(f"- Categorical features (one-hot encoded): {len(categorical_feature_names)}")
print(f"- Total features: {len(all_feature_names)}")

X_train_df = pd.DataFrame(X_train_transformed, columns=all_feature_names)
X_test_df = pd.DataFrame(X_test_transformed, columns=all_feature_names)

print("\nNumeric columns statistics (after scaling):")
print(X_train_df[numeric_columns].describe().round(6))

# Sample of transformed data
print("\nSample of transformed data (first 3 rows, first 10 columns):")
print(X_train_df.iloc[:3, :10].round(3))


AFTER TRANSFORMATION
Training set shape: (36177, 96)
Test set shape: (9045, 96)
Features increased from 14 to 96

Feature breakdown:
- Numeric features (scaled): 6
- Categorical features (one-hot encoded): 90
- Total features: 96

Numeric columns statistics (after scaling):
                age        fnlwgt  education-num  capital-gain  capital-loss  \
count  36177.000000  36177.000000   36177.000000  36177.000000  36177.000000   
mean      -0.000000      0.000000       0.000000      0.000000      0.000000   
std        1.000014      1.000014       1.000014      1.000014      1.000014   
min       -1.629169     -1.665969      -3.568925     -0.144863     -0.217040   
25%       -0.797894     -0.684164      -0.437290     -0.144863     -0.217040   
50%       -0.117760     -0.109115      -0.045836     -0.144863     -0.217040   
75%        0.637944      0.454790       1.128527     -0.144863     -0.217040   
max        3.887473     12.292796       2.302891     13.153288     10.627147   

   